### intention detection using Subcat method for both Episode model and Commit list

In [20]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Dict, List, Tuple, Set
from collections import defaultdict

import pandas as pd

# ---- Optional stemming ----
try:
    from nltk.stem.snowball import SnowballStemmer
    _stemmer = SnowballStemmer("english")

    def _stem(token: str) -> str:
        return _stemmer.stem(token)
except Exception:
    def _stem(token: str) -> str:
        return token


# ============================================================
# I/O (your paths)
# ============================================================
BASE_DIR = Path(
    r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_1_Intention"
)

# Episode-level dataset
EPISODES_CSV = BASE_DIR / "2_All_episodes_with_messages.csv"

# Commit-level dataset (all boundary commits as rows)
COMMITS_CSV = BASE_DIR / "1_All_Commits_PR_Msg_Iss.csv"

# Dictionary (lemmatized/long format)
DICTIONARY_CSV = BASE_DIR / "0_dictionary_lemmatized.csv"

# Outputs
OUTPUT_DIR = BASE_DIR / "Output"
OUT_EPISODES = OUTPUT_DIR / "2_All_episodes_with_messages_with_intentions_subcat.csv"
OUT_COMMITS = OUTPUT_DIR / "1_All_Commits_PR_Msg_Iss_with_intentions_subcat.csv"


# ============================================================
# Boost coefficients
#   (keep these aligned with your intended weighting)
# ============================================================
ISSUE_BOOST = .0
COMMIT_BOOST = 1.0
PR_BOOST = 1.0


# ============================================================
# Scoring / selection params
# ============================================================
MIN_SCORE = 1.5
MULTI_RATIO = 0.80
MAX_LABELS = 3

CONF_W_SHARE = 0.6
CONF_W_MARGIN = 0.4

HIGH_TH = 0.75
MED_TH = 0.55


# ============================================================
# Episode schema
# ============================================================
END_COMMIT_COL = "episode_end_commit_sha"


# ============================================================
# Text columns used for intention detection
#   EPISODES: uses start_* and end_* versions (8 context columns each boundary)
# ============================================================
EP_START_PARTS: List[Tuple[List[str], float]] = [
    (["start_commit_subject", "start_commit_body"], COMMIT_BOOST),

    (["start_pr_titles"], PR_BOOST),
    (["start_pr_bodies"], PR_BOOST),
    (["start_pr_comments_and_reviews"], PR_BOOST),

    (["start_issue_titles"], ISSUE_BOOST),
    (["start_issue_bodies"], ISSUE_BOOST),
    (["start_issue_comments"], ISSUE_BOOST),
]

EP_END_PARTS: List[Tuple[List[str], float]] = [
    (["end_commit_subject", "end_commit_body"], COMMIT_BOOST),

    (["end_pr_titles"], PR_BOOST),
    (["end_pr_bodies"], PR_BOOST),
    (["end_pr_comments_and_reviews"], PR_BOOST),

    (["end_issue_titles"], ISSUE_BOOST),
    (["end_issue_bodies"], ISSUE_BOOST),
    (["end_issue_comments"], ISSUE_BOOST),
]


# ============================================================
# Commit-level dataset columns (8 context columns, no start_/end_ prefix)
# ============================================================
COMMIT_PARTS: List[Tuple[List[str], float]] = [
    (["commit_subject", "commit_body"], COMMIT_BOOST),

    (["pr_titles"], PR_BOOST),
    (["pr_bodies"], PR_BOOST),
    (["pr_comments_and_reviews"], PR_BOOST),

    (["issue_titles"], ISSUE_BOOST),
    (["issue_bodies"], ISSUE_BOOST),
    (["issue_comments"], ISSUE_BOOST),
]


# ============================================================
# Helpers
# ============================================================
def _safe_str(v) -> str:
    if v is None:
        return ""
    if isinstance(v, float) and pd.isna(v):
        return ""
    s = str(v).strip()
    return "" if s.lower() == "nan" else s


def normalize_text_for_tokens(text: str) -> str:
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)  # de-camelcase
    text = text.replace("_", " ").replace("-", " ")
    return text.lower()


def tokenize_and_stem(text: str) -> List[str]:
    """
    Keeps digits (e2e, v2, aab, etc). Stems only purely alphabetic tokens.
    """
    if not text:
        return []
    text = normalize_text_for_tokens(text)
    raw = re.findall(r"[a-z0-9]+", text)
    out: List[str] = []
    for t in raw:
        if any(ch.isdigit() for ch in t):
            out.append(t)
        else:
            out.append(_stem(t))
    return out


def count_phrase_occurrences(tokens: List[str], phrase_tokens: List[str]) -> int:
    if not phrase_tokens or not tokens:
        return 0
    if len(phrase_tokens) == 1:
        p = phrase_tokens[0]
        return sum(1 for t in tokens if t == p)
    n = len(phrase_tokens)
    cnt = 0
    for i in range(len(tokens) - n + 1):
        if tokens[i: i + n] == phrase_tokens:
            cnt += 1
    return cnt


def build_parts_from_row(row: pd.Series, spec: List[Tuple[List[str], float]]) -> List[Tuple[str, float]]:
    parts: List[Tuple[str, float]] = []
    for cols, mult in spec:
        txt = "\n".join([_safe_str(row.get(c, "")) for c in cols]).strip()
        if txt:
            parts.append((txt, mult))
    return parts


# ============================================================
# Dictionary loading (LONG format)
# ============================================================
def load_dictionary_long(path: Path) -> Tuple[Dict[str, List[Dict]], List[Dict]]:
    """
    Expects columns: label, keyword, optional weight, optional group.
    If label == "Blacklist" (case-insensitive), goes to blacklist.
    """
    df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8", engine="python")

    colmap = {str(c).strip().lower(): c for c in df.columns}
    if "label" not in colmap or "keyword" not in colmap:
        raise ValueError("Dictionary CSV must have columns: label, keyword")

    label_col = colmap["label"]
    keyword_col = colmap["keyword"]
    weight_col = colmap.get("weight", None)
    group_col = colmap.get("group", None)

    dict_terms: Dict[str, List[Dict]] = defaultdict(list)
    blacklist_terms: List[Dict] = []

    for _, row in df.iterrows():
        lab = _safe_str(row.get(label_col, "")).strip()
        kw = _safe_str(row.get(keyword_col, "")).strip()
        if not lab or not kw:
            continue

        wt = 1.0
        if weight_col is not None:
            try:
                wt = float(row.get(weight_col, 1.0))
            except Exception:
                wt = 1.0

        grp = _safe_str(row.get(group_col, "")).strip().upper() if group_col is not None else ""

        stem_tokens = tokenize_and_stem(kw)
        if not stem_tokens:
            continue

        item = {"keyword": kw, "weight": wt, "stem_tokens": stem_tokens, "group": grp}

        if lab.lower() == "blacklist":
            blacklist_terms.append(item)
        else:
            dict_terms[lab].append(item)

    # de-dup by stem sequence within label
    def dedup(items: List[Dict]) -> List[Dict]:
        seen = set()
        out = []
        for it in items:
            k = " ".join(it["stem_tokens"])
            if k in seen:
                continue
            seen.add(k)
            out.append(it)
        return out

    dict_terms = {lab: dedup(items) for lab, items in dict_terms.items()}
    blacklist_terms = dedup(blacklist_terms)

    if not dict_terms:
        raise ValueError("No label keywords loaded from dictionary CSV")

    return dict_terms, blacklist_terms


# ============================================================
# Constraints using GLOBAL groups
# ============================================================
def passes_constraints(label: str, global_groups: Set[str]) -> bool:
    if label == "Introduce / strengthen CI-backed tests":
        return ("CI" in global_groups) and ("TEST" in global_groups)

    if label == "Migrate or modernise CI infrastructure":
        return ("MIGRATE" in global_groups) and ("CI" in global_groups)

    if label == "Clean up or simplify CI / environment configuration":
        return ("CLEANUP" in global_groups) and (("CI" in global_groups) or ("ENV" in global_groups))

    # Perf/Stability only when CI/TEST stability signal exists
    if label == "Address performance or stability issues":
        return (("PERF" in global_groups) or ("FAIL" in global_groups)) and (("CI" in global_groups) or ("TEST" in global_groups))

    return True


# ============================================================
# Scoring
# ============================================================
def classify_parts(
    parts: List[Tuple[str, float]],
    dict_terms: Dict[str, List[Dict]],
    blacklist_terms: List[Dict],
    cap_per_keyword_per_part: bool = True,
) -> Tuple[Dict[str, float], Dict[str, List[str]], List[str], Set[str]]:
    """
    Returns:
      scores[label] = float
      matched_keywords[label] = [kw...]
      blacklist_hits = [kw...]
      global_groups = {GROUP...} across all matches
    """
    token_parts: List[Tuple[List[str], float]] = []
    for text, mult in parts:
        toks = tokenize_and_stem(text)
        if toks:
            token_parts.append((toks, mult))

    scores = {label: 0.0 for label in dict_terms}
    matched_keywords: Dict[str, List[str]] = {label: [] for label in dict_terms}

    # blacklist hits
    blacklist_hits: List[str] = []
    for it in blacklist_terms:
        for toks, _mult in token_parts:
            if count_phrase_occurrences(toks, it["stem_tokens"]) > 0:
                blacklist_hits.append(it["keyword"])
                break
    blacklist_hits = sorted(set(blacklist_hits))

    global_groups: Set[str] = set()

    # label scoring
    for label, items in dict_terms.items():
        for it in items:
            total_occ = 0.0
            hit = False
            for toks, mult in token_parts:
                occ = count_phrase_occurrences(toks, it["stem_tokens"])
                if occ > 0:
                    hit = True
                    if cap_per_keyword_per_part:
                        occ = 1
                    total_occ += (occ * mult)

            if hit:
                scores[label] += it["weight"] * total_occ
                matched_keywords[label].append(it["keyword"])

                g = (it.get("group") or "").strip().upper()
                if g:
                    global_groups.add(g)

    # Apply constraints based on global groups
    for label in list(scores.keys()):
        if not passes_constraints(label, global_groups):
            scores[label] = 0.0
            matched_keywords[label] = []

    return scores, matched_keywords, blacklist_hits, global_groups


def apply_precedence(scores: Dict[str, float], matched_keywords: Dict[str, List[str]], global_groups: Set[str]) -> None:
    """
    If stability evidence (PERF or FAIL) appears alongside CI/TEST,
    prefer PERF/Stability over CI-backed-tests.
    """
    perf_lab = "Address performance or stability issues"
    ci_lab = "Introduce / strengthen CI-backed tests"

    has_stability_signal = ("PERF" in global_groups) or ("FAIL" in global_groups)
    has_ci_or_test = ("CI" in global_groups) or ("TEST" in global_groups)

    if has_stability_signal and has_ci_or_test:
        # ---- FIX: guard against KeyError if label isn't present in dictionary ----
        if ci_lab in scores:
            scores[ci_lab] = 0.0
            matched_keywords[ci_lab] = []

        if scores.get(perf_lab, 0.0) > 0:
            scores[perf_lab] *= 1.5


def assign_labels_multi(
    scores: Dict[str, float],
    matched_keywords: Dict[str, List[str]],
    blacklist_hits: List[str],
    min_score: float = MIN_SCORE,
    multi_ratio: float = MULTI_RATIO,
    max_labels: int = MAX_LABELS,
) -> Dict[str, object]:
    items = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    top_label, top_score = items[0] if items else ("", 0.0)
    second_label, second_score = items[1] if len(items) > 1 else ("", 0.0)

    total = float(sum(scores.values()))
    if top_score < min_score:
        return {
            "label_str": None,
            "top_label": top_label, "top_score": float(top_score),
            "second_label": second_label, "second_score": float(second_score),
            "total_score": float(total),
            "selected_score_sum": 0.0,
            "confidence_share_selected": 0.0,
            "confidence_margin_selected": 0.0,
            "confidence_value": 0.0,
            "confidence_level": "UNLABELED_NOISE" if blacklist_hits else "UNLABELED",
            "matched_keywords": "",
            "blacklist_hits": "; ".join(blacklist_hits),
        }

    chosen: List[str] = []
    chosen_scores: List[float] = []

    for lab, sc in items:
        if sc < min_score:
            break
        if sc >= top_score * multi_ratio:
            chosen.append(lab)
            chosen_scores.append(sc)
        if len(chosen) >= max_labels:
            break

    chosen_set = set(chosen)
    next_unselected = 0.0
    for lab, sc in items:
        if lab not in chosen_set:
            next_unselected = sc
            break

    selected_sum = float(sum(chosen_scores))
    share_selected = (selected_sum / total) if total > 0 else 0.0
    min_selected = float(min(chosen_scores)) if chosen_scores else 0.0
    margin_selected = ((min_selected - next_unselected) / min_selected) if min_selected > 0 else 0.0
    margin_selected = max(0.0, min(1.0, margin_selected))

    confidence_value = (CONF_W_SHARE * share_selected) + (CONF_W_MARGIN * margin_selected)

    if confidence_value >= HIGH_TH and selected_sum >= 2.5:
        level = "HIGH"
    elif confidence_value >= MED_TH:
        level = "MEDIUM"
    else:
        level = "LOW"

    matched = sorted({kw for lab in chosen for kw in matched_keywords.get(lab, [])})

    return {
        "label_str": " || ".join(chosen),
        "top_label": top_label, "top_score": float(top_score),
        "second_label": second_label, "second_score": float(second_score),
        "total_score": float(total),
        "selected_score_sum": float(selected_sum),
        "confidence_share_selected": float(share_selected),
        "confidence_margin_selected": float(margin_selected),
        "confidence_value": float(confidence_value),
        "confidence_level": level,
        "matched_keywords": "; ".join(matched),
        "blacklist_hits": "; ".join(blacklist_hits),
    }


def label_boundary(row: pd.Series, parts_spec, dict_terms, blacklist_terms) -> Dict[str, object]:
    parts = build_parts_from_row(row, parts_spec)
    scores, matched_keywords, blacklist_hits, global_groups = classify_parts(parts, dict_terms, blacklist_terms)
    apply_precedence(scores, matched_keywords, global_groups)
    return assign_labels_multi(scores, matched_keywords, blacklist_hits)


def _warn_missing_cols(df: pd.DataFrame, required: List[str], name: str) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f"[warn] {name}: missing expected columns:")
        for c in missing:
            print("  -", c)


# ============================================================
# Main
# ============================================================
def main() -> None:
    if not EPISODES_CSV.exists():
        raise FileNotFoundError(f"Episodes CSV not found: {EPISODES_CSV}")
    if not COMMITS_CSV.exists():
        raise FileNotFoundError(f"Commits CSV not found: {COMMITS_CSV}")
    if not DICTIONARY_CSV.exists():
        raise FileNotFoundError(f"Dictionary CSV not found: {DICTIONARY_CSV}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    dict_terms, blacklist_terms = load_dictionary_long(DICTIONARY_CSV)

    # ----------------------------
    # 1) EPISODE-LEVEL DETECTION
    # ----------------------------
    ep = pd.read_csv(EPISODES_CSV, dtype=str, keep_default_na=False, encoding="utf-8", engine="python")

    _warn_missing_cols(
        ep,
        required=[
            END_COMMIT_COL,
            "start_commit_subject", "start_commit_body",
            "start_pr_titles", "start_pr_bodies", "start_pr_comments_and_reviews",
            "start_issue_titles", "start_issue_bodies", "start_issue_comments",
            "end_commit_subject", "end_commit_body",
            "end_pr_titles", "end_pr_bodies", "end_pr_comments_and_reviews",
            "end_issue_titles", "end_issue_bodies", "end_issue_comments",
        ],
        name="EPISODES_CSV",
    )

    start_results = []
    end_results = []

    for _, row in ep.iterrows():
        # start: always label
        s = label_boundary(row, EP_START_PARTS, dict_terms, blacklist_terms)
        start_results.append(s)

        # end: only if end commit exists (works for empty strings too)
        end_sha = _safe_str(row.get(END_COMMIT_COL, ""))
        if not end_sha.strip():
            end_results.append({
                "label_str": None,
                "top_label": "", "top_score": 0.0,
                "second_label": "", "second_score": 0.0,
                "total_score": 0.0,
                "selected_score_sum": 0.0,
                "confidence_share_selected": 0.0,
                "confidence_margin_selected": 0.0,
                "confidence_value": 0.0,
                "confidence_level": "NO_END_COMMIT",
                "matched_keywords": "",
                "blacklist_hits": "",
            })
        else:
            e = label_boundary(row, EP_END_PARTS, dict_terms, blacklist_terms)
            end_results.append(e)

    ep_out = pd.concat(
        [ep, pd.DataFrame(start_results).add_prefix("start_"), pd.DataFrame(end_results).add_prefix("end_")],
        axis=1
    )
    ep_out.to_csv(OUT_EPISODES, index=False, encoding="utf-8")
    print("[ok] wrote episodes intentions:", OUT_EPISODES)

    # ----------------------------
    # 2) COMMIT-LEVEL DETECTION
    # ----------------------------
    cm = pd.read_csv(COMMITS_CSV, dtype=str, keep_default_na=False, encoding="utf-8", engine="python")

    _warn_missing_cols(
        cm,
        required=[
            "repo_name", "commit_sha",
            "commit_subject", "commit_body",
            "pr_titles", "pr_bodies", "pr_comments_and_reviews",
            "issue_titles", "issue_bodies", "issue_comments",
        ],
        name="COMMITS_CSV",
    )

    cm_results = []
    for _, row in cm.iterrows():
        r = label_boundary(row, COMMIT_PARTS, dict_terms, blacklist_terms)
        cm_results.append(r)

    cm_int = pd.DataFrame(cm_results).add_prefix("intent_")
    cm_out = pd.concat([cm, cm_int], axis=1)
    cm_out.to_csv(OUT_COMMITS, index=False, encoding="utf-8")
    print("[ok] wrote commit intentions:", OUT_COMMITS)

    print("[done] episodes_rows =", len(ep_out), " | commits_rows =", len(cm_out))


if __name__ == "__main__":
    main()


[ok] wrote episodes intentions: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_1_Intention\Output\2_All_episodes_with_messages_with_intentions_subcat.csv
[ok] wrote commit intentions: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_1_Intention\Output\1_All_Commits_PR_Msg_Iss_with_intentions_subcat.csv
[done] episodes_rows = 535  | commits_rows = 535


In [5]:
#filter for sweet spot phrases only

In [9]:
from __future__ import annotations

import re
from pathlib import Path
import pandas as pd

# ============================================================
# Paths (match your folder layout)
# ============================================================
BASE_DIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_1_Intention")
OUTPUT_DIR = BASE_DIR / "Output"

CAND_IN = OUTPUT_DIR / "dictionary_candidate_suggestions_ngrams_idf.csv"
CAND_OUT = OUTPUT_DIR / "dictionary_candidate_suggestions_TOP20.csv"

# ============================================================
# Strategy thresholds (tune if needed)
# ============================================================
TOP_K = 20

# "Sweet spot" band (good balance between generality + discrimination)
SWEET_MIN = 0.01     # ~1% of boundaries
SWEET_MAX = 0.15     # ~15% of boundaries

# Above this tends to be too generic for dictionary labels
GENERIC_MAX = 0.25   # ~25% of boundaries

# Require some repetition to reduce one-off noise
MIN_DF_DOCS = 5

# ============================================================
# Noise filtering (keeps codecov io / sonarcloud io, removes URL-ish junk)
# ============================================================
STOPWORDS = {
    "a","an","the","and","or","to","of","in","on","for","with","by","from","as","at","it","is","are","be",
    "this","that","these","those","we","you","they","i","our","your","their","was","were","will","can",
    "not","no","yes","if","then","else","when","while","into","about","over","under","up","down","out",
    "more","most","some","any","all","one","two","three","use","using","used"
}

# Tokens that often indicate boilerplate/URL/tracking noise in mined text
NOISE_TOKENS = {
    "http","https","www","utm","utm_source","utm_medium","utm_campaign","utm_term","utm_content",
    "href","src","png","jpg","jpeg","gif","svg","html","php"
}

_alnum_re = re.compile(r"^[a-z0-9]+$")

def looks_like_noise_phrase(phrase: str) -> bool:
    p = (phrase or "").strip().lower()
    if not p:
        return True

    toks = p.split()
    if not toks:
        return True

    # If tokens contain obvious URL/tracking artifacts
    if any(t in NOISE_TOKENS for t in toks):
        return True

    # If any token is not alnum (should be rare if your tokenizer already normalized,
    # but this protects against unexpected punctuation)
    if any(not _alnum_re.match(t) for t in toks):
        return True

    # Stopword-only phrases are almost never useful as dictionary keywords
    if all(t in STOPWORDS for t in toks):
        return True

    # Mostly numeric phrases tend to be noise (versions/build numbers)
    num_ratio = sum(1 for t in toks if t.isdigit()) / len(toks)
    if num_ratio >= 0.67:
        return True

    # Very short phrases can be noisy unless meaningful; keep 2+ tokens already by design,
    # but this is a safeguard.
    if len(toks) < 2:
        return True

    return False

# ============================================================
# Selection logic: Sweet spot first, then moderate, then rare (if needed)
# ============================================================
def select_top_k(df: pd.DataFrame, k: int = TOP_K) -> pd.DataFrame:
    # Basic hygiene + types
    for col in ["score", "tf_weighted", "df_docs", "doc_count_N", "df_ratio"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["candidate_phrase", "score", "df_docs", "doc_count_N", "df_ratio"])
    df["candidate_phrase"] = df["candidate_phrase"].astype(str)
    df["boundary"] = df["boundary"].astype(str)

    # Remove noise candidates
    df = df[~df["candidate_phrase"].map(looks_like_noise_phrase)].copy()

    # Remove too-generic candidates (keeps useful platform/tool terms up to GENERIC_MAX)
    df = df[df["df_ratio"] <= GENERIC_MAX].copy()

    # Require some repetition
    df = df[df["df_docs"] >= MIN_DF_DOCS].copy()

    # Rank buckets
    sweet = df[(df["df_ratio"] >= SWEET_MIN) & (df["df_ratio"] <= SWEET_MAX)].copy()
    moderate = df[(df["df_ratio"] > SWEET_MAX) & (df["df_ratio"] <= GENERIC_MAX)].copy()
    rare = df[df["df_ratio"] < SWEET_MIN].copy()

    # Sort each bucket by score desc, then tf_weighted desc, then df_docs desc
    sort_cols = ["score", "tf_weighted", "df_docs", "df_ratio"]
    sweet = sweet.sort_values(sort_cols, ascending=[False, False, False, True])
    moderate = moderate.sort_values(sort_cols, ascending=[False, False, False, True])
    rare = rare.sort_values(sort_cols, ascending=[False, False, False, True])

    # Combine in priority order; de-dup by phrase (keep best)
    combined = pd.concat([sweet, moderate, rare], ignore_index=True)
    combined = combined.drop_duplicates(subset=["candidate_phrase"], keep="first")

    return combined.head(k)

def main() -> None:
    if not CAND_IN.exists():
        raise FileNotFoundError(f"Input suggestions file not found: {CAND_IN}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(CAND_IN, dtype=str, keep_default_na=False, encoding="utf-8", engine="python")

    required = {"boundary","candidate_phrase","score","tf_weighted","df_docs","doc_count_N","df_ratio"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {CAND_IN}: {sorted(missing)}")

    top = select_top_k(df, k=TOP_K)

    # Keep a clean column order
    cols = [
        "boundary","candidate_phrase","score","tf_weighted","df_docs","doc_count_N","df_ratio",
        "example_repo_name","example_episode_index"
    ]
    cols = [c for c in cols if c in top.columns]
    top = top[cols]

    top.to_csv(CAND_OUT, index=False, encoding="utf-8")
    print(f"[ok] wrote top-{TOP_K} filtered candidates: {CAND_OUT}")
    print(f"[info] rows_out={len(top)} (may be < {TOP_K} if filters are strict)")

if __name__ == "__main__":
    main()


[ok] wrote top-20 filtered candidates: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_1_Intention\Output\dictionary_candidate_suggestions_TOP20.csv
[info] rows_out=20 (may be < 20 if filters are strict)
